In [ ]:
# Find the gold index by matching the target text to arguments


def get_gold_idx(sample: dict) -> int:
    """
    Resolves the gold answer index into resps, handling both
    integer-index targets and arbitrary string targets.
    """
    # return sample["mcc"][0]
    
    # return sample["mcc"][0]
    
    if "mcc" in sample:
        gold, acc = sample["mcc"]
        if isinstance(gold, int):
            return gold
    
    if "gold_index" in sample:
        return sample["gold_index"]

    target = sample["target"]
    n = len(sample["resps"])

    # Case 1: integer index (stored as int or castable string like "2")
    try:
        idx = int(target)
        if 0 <= idx < n:
            return idx
    except (ValueError, TypeError):
        pass

    # Case 2: string target — match against the continuation in arguments
    target_str = str(target).strip()
    for i, arg in enumerate(sample["arguments"]):
        continuation = arg[1].strip()
        if continuation == target_str:
            return i

    raise ValueError(f"Could not resolve target {target!r} to any of: {[a[1] for a in sample['arguments']]}")


def get_logprob_pred_index(s):
    # return index of max logprob among filtered_resps

    return max(range(len(s["filtered_resps"])), key=lambda i: s["filtered_resps"][i][0])

In [ ]:
import pprint


# NOTE: works for loglikelihood
# NOTE: doesnt work sometimes for multiple_choices
def validate_sample(s) -> bool:
    our_is_correct = get_gold_idx(s) == get_logprob_pred_index(s)
    their_is_correct = s["acc"] == 1
    if our_is_correct != their_is_correct:
        pprint.pprint(s)
        print(f"Our prediction: {our_is_correct}, Their acc: {their_is_correct}")
        print(f"Our golden idx: {get_gold_idx(s)}, Our pred idx: {get_logprob_pred_index(s)}")
        return False
    return True

In [2]:
def get_output_type(data: dict, bench_name: str) -> str:
    return data["configs"][bench_name]["output_type"]

In [8]:
# list all files in "../clean_results/greedy/gemma-2b-it/benchmarks"
# open the data and create a list of file names which contain a benchmark with output_type "loglikelihood" or "multiple_choice"

import json
import os

def find_benchmarks_with_output_type(output_type: str, dir_path: str) -> list:
    matching_benchmarks = []
    for root, dirs, files in os.walk(dir_path):
        for file in files:
            if file.endswith(".json"):
                with open(os.path.join(root, file), "r") as f:
                    data = json.load(f)
                    for bench_name in data["configs"]:
                        if get_output_type(data, bench_name) == output_type:
                            # get only the last part without the extension
                            matching_benchmarks.append(file.split(".")[0])
                            break
    return matching_benchmarks


res_loglikelihood = find_benchmarks_with_output_type("loglikelihood", "../clean_results/greedy/gemma-2b-it/benchmarks")
res_multi = find_benchmarks_with_output_type("multiple_choice", "../clean_results/greedy/gemma-2b-it/benchmarks")
res_gen = find_benchmarks_with_output_type("generate_until", "../clean_results/greedy/gemma-2b-it/benchmarks")
res_rolling = find_benchmarks_with_output_type("loglikelihood_rolling", "../clean_results/greedy/gemma-2b-it/benchmarks")

# print counts

print(len(res_loglikelihood))
print(len(res_multi))
print(len(res_gen))
print(len(res_rolling))


0
11
9
1


In [10]:
print(res_multi)

['anli', 'blimp', 'mastermind_easy', 'metabench_arc', 'metabench_hellaswag', 'metabench_mmlu', 'metabench_truthfulqa', 'metabench_winogrande', 'piqa', 'toxigen', 'wmdp']


In [ ]:
import json
from tqdm.auto import tqdm
import os

main_bench_names = [
    "anli",
    "blimp",
    "ifeval",
    "jsonschema_bench",
    "mastermind_easy",
    "mbpp",
    "metabench_arc",
    "metabench_gsm8k",
    "metabench_hellaswag",
    "metabench_mmlu",
    "metabench_truthfulqa",
    "metabench_winogrande",
    "piqa",
    "toxigen",
    "wikitext",
    "wmdp",
    "xquad_ar",
    "xquad_en",
    "xquad_es",
    "xquad_ru",
    "xquad_zh",
]
for main_name in tqdm(main_bench_names):
    path = f"../test-Qwen2.5-0.5B-Instruct/benchmarks/{main_name}.json"

    # check if path exists, if no then log and skip
    if not os.path.exists(path):
        print(f"Path {path} does not exist, skipping.")
        continue

    with open(path, "r") as f:
        data = json.load(f)

    bench_names = list(data["samples"].keys())
    for bench_name in tqdm(bench_names):
        print(bench_name)
        output = get_output_type(data, bench_name)
        if output != "loglikelihood" and output != "multiple_choice":
            print(f"Skipping {bench_name} because output type is not logprobs but {output}")
            continue

        samples = data["samples"][bench_name]
        for i, s in enumerate(samples):
            if not validate_sample(s):
                print(f"Failed for benchmark {bench_name} sample {i}, continuing to next benchmark")
                break

In [ ]:
print()